## 0. Initialize and Load Data Frames

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Load Delta tables into pure PySpark DataFrames
df_customers = spark.table("retail_sales_analytics.analytics.dim_customers")
df_products = spark.table("retail_sales_analytics.analytics.dim_products")
df_fact_sales = spark.table("retail_sales_analytics.analytics.dim_fact_sales")

print("DataFrames loaded successfully for EDA!")

## 1. Database and Schema Exploration
*Investigating database metadata, schemas, and structure.*

In [0]:
print("--- Customers Table Schema ---")
df_customers.printSchema()

print("--- Products Table Schema ---")
df_products.printSchema()

print("--- Fact Sales Table Schema ---")
df_fact_sales.printSchema()

## 3. Dimensions Exploration
*Identifying unique values, categories, and hierarchies.*

In [0]:
# Assign dimension transformations to clean variables
unique_countries_df = df_customers.select("country").distinct().orderBy("country")
unique_categories_df = df_products.select("category").distinct().orderBy("category")
product_hierarchy_df = df_products.select("category", "subcategory", "product_name")\
    .distinct()\
    .orderBy("category", "subcategory", "product_name")

print("--- Unique Countries ---")
display(unique_countries_df)

print("--- Unique Categories Only ---")
display(unique_categories_df)

print("--- Unique Product Hierarchy ---")
display(product_hierarchy_df)

## 2. Date Range Exploration
*Determining temporal boundaries and historical scope.*

In [0]:
# Find transaction date range and span
date_range_df = df_fact_sales.select(
    F.min("order_date").alias("first_order_date"),
    F.max("order_date").alias("last_order_date"),
    
    # Duration in years (subtracting years or months / 12)
    (F.year(F.max("order_date")) - F.year(F.min("order_date"))).alias("order_range_years"),
    
    # Duration in months using months_between
    F.round(F.months_between(F.max("order_date"), F.min("order_date")), 1).alias("order_range_months"),
    
    # Duration in days using datediff
    F.datediff(F.max("order_date"), F.min("order_date")).alias("order_range_days")
)

display(date_range_df)


# Find oldest and youngest customer birthdate boundaries
dob_range_df = df_customers.select(
    F.min("birthdate").alias("oldest_birthdate"),
    F.max("birthdate").alias("youngest_birthdate")
)

display(dob_range_df)

## 4. Measure Exploration
*Calculating key quantitative metrics of the business[cite: 6].*

In [0]:
from pyspark.sql.functions import *

# 1. Total Sales metric
metric_sales_df = df_fact_sales.select(
    lit("Total_Sales").alias("measure_name"), 
    sum("sales_amount").cast("double").alias("measure_value")
)

# 2. Total Quantity metric
metric_qty_df = df_fact_sales.select(
    lit("Total_Quantity").alias("measure_name"), 
    sum("quantity").cast("double").alias("measure_value")
)

# 3. Average Price metric (rounded to 2 decimal places — change to 0 if you want exact integer format like 486)
metric_price_df = df_fact_sales.select(
    lit("Average Price").alias("measure_name"), 
    round(avg("price"), 2).cast("double").alias("measure_value")
)

# 4. Total Orders metric
metric_orders_df = df_fact_sales.select(
    lit("Total Nr.Orders").alias("measure_name"), 
    countDistinct("order_number").cast("double").alias("measure_value")
)

# 5. Total Products metric
metric_products_df = df_products.select(
    lit("Total Nr.Products").alias("measure_name"), 
    count("product_name").cast("double").alias("measure_value")
)

# 6. Total Customers metric
metric_customers_df = df_customers.select(
    lit("Total Nr.Customers").alias("measure_name"), 
    count("customer_key").cast("double").alias("measure_value")
)

# Combine all individual metric DataFrames into a single executive report variable
executive_report_df = (
    metric_sales_df
    .unionByName(metric_qty_df)
    .unionByName(metric_price_df)
    .unionByName(metric_orders_df)
    .unionByName(metric_products_df)
    .unionByName(metric_customers_df)
)

print("--- Executive Business Report ---")
display(executive_report_df)

## Magnitude Analysis
*Comparing measure values by categories and dimensions to understand size, scale, and distribution[cite: 5].*

In [0]:
from pyspark.sql.functions import *

# 1. Find total customers by countries
customers_by_country_df = (
    df_customers
    .groupBy("country")
    .agg(count("customer_key").alias("Total_customers"))
    .orderBy(col("Total_customers").desc())
)

# 2. Find total customers by gender
customers_by_gender_df = (
    df_customers
    .groupBy("gender")
    .agg(count("customer_key").alias("total_customers"))
    .orderBy(col("total_customers").desc())
)

# 3. Find total products by category
products_by_category_df = (
    df_products
    .groupBy("category")
    .agg(count("product_key").alias("Total_Products"))
    .orderBy(col("Total_Products").desc())
)

# 4. What is the average costs in each category? (Casted to int to match SQL Server truncation)
avg_costs_by_category_df = (
    df_products
    .groupBy("category")
    .agg(avg("cost").cast("int").alias("Avg_Costs"))
    .orderBy(col("Avg_Costs").desc())
)

# 5. What is the total revenue generated for each category?
revenue_by_category_df = (
    df_fact_sales
    .join(df_products, "product_key", "left")
    .groupBy("category")
    .agg(sum("sales_amount").alias("Total_Revenue"))
    .orderBy(col("Total_Revenue").desc())
)

# 6. What is the total revenue generated by each customer?
revenue_by_customer_df = (
    df_fact_sales
    .join(df_customers, "customer_key", "left")
    .groupBy("customer_key", "first_name", "last_name")
    .agg(sum("sales_amount").alias("Total_revenue"))
    .orderBy(col("Total_revenue").desc())
)

# 7. What is the distribution of sold items across countries?
items_by_country_df = (
    df_fact_sales
    .join(df_customers, "customer_key", "left")
    .groupBy("country")
    .agg(sum("quantity").alias("Total_Sold_Items"))
    .orderBy(col("Total_Sold_Items").desc())
)

# --- Display Results ---
print("--- 1. Total Customers by Country ---")
display(customers_by_country_df)

print("--- 2. Total Customers by Gender ---")
display(customers_by_gender_df)

print("--- 3. Total Products by Category ---")
display(products_by_category_df)

print("--- 4. Average Costs in Each Category ---")
display(avg_costs_by_category_df)

print("--- 5. Total Revenue by Category ---")
display(revenue_by_category_df)

print("--- 6. Total Revenue by Customer ---")
display(revenue_by_customer_df)

print("--- 7. Sold Items Distribution Across Countries ---")
display(items_by_country_df)

## Ranking Analysis
*Ordering the values of dimensions by measures to identify Top N and Bottom N performers[cite: 7].*

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

# -------------------------------------------------------------
# 1. Which 5 products generating the highest revenue? (TOP 5)
# -------------------------------------------------------------
top_5_products_df = (
    df_fact_sales.join(df_products, "product_key", "left")
    .groupBy("product_name")
    .agg(sum("sales_amount").alias("Total_Revenue"))
    .orderBy(col("Total_Revenue").desc())
    .limit(5)
)

# -------------------------------------------------------------
# 2. Which 5 subcategories generating the highest revenue? (TOP 5)
# -------------------------------------------------------------
top_5_subcategories_df = (
    df_fact_sales.join(df_products, "product_key", "left")
    .groupBy("subcategory")
    .agg(sum("sales_amount").alias("Total_Revenue"))
    .orderBy(col("Total_Revenue").desc())
    .limit(5)
)

# -------------------------------------------------------------
# 3. USING WINDOW FUNCTION: Top 5 products by revenue
# -------------------------------------------------------------
product_spec = Window.orderBy(col("Total_Revenue").desc())
top_5_products_window_df = (
    df_fact_sales.join(df_products, "product_key", "left")
    .groupBy("product_name")
    .agg(sum("sales_amount").alias("Total_Revenue"))
    .withColumn("Rank_Products", rank().over(product_spec))
    .filter(col("Rank_Products") <= 5)
)

# -------------------------------------------------------------
# 4. What are the 5 worst-performing products in terms of sales? (Bottom 5)
# -------------------------------------------------------------
worst_5_products_df = (
    df_fact_sales.join(df_products, "product_key", "left")
    .groupBy("product_name")
    .agg(sum("sales_amount").alias("Total_Revenue"))
    .orderBy(col("Total_Revenue").asc())
    .limit(5)
)

# -------------------------------------------------------------
# 5. Find the top 10 customers who have generated the highest revenue (TOP 10)
# -------------------------------------------------------------
top_10_customers_df = (
    df_fact_sales.join(df_customers, "customer_key", "left")
    .groupBy("customer_key", "first_name", "last_name")
    .agg(sum("sales_amount").alias("Total_revenue"))
    .orderBy(col("Total_revenue").desc())
    .limit(10)
)

# -------------------------------------------------------------
# 6. USING WINDOW FUNCTION: Top 10 customers by revenue
# -------------------------------------------------------------
customer_rev_spec = Window.orderBy(col("Total_revenue").desc())
top_10_customers_window_df = (
    df_fact_sales.join(df_customers, "customer_key", "left")
    .groupBy("customer_key", "first_name", "last_name")
    .agg(sum("sales_amount").alias("Total_revenue"))
    .withColumn("Rank_customer", rank().over(customer_rev_spec))
    .filter(col("Rank_customer") <= 10)
)

# -------------------------------------------------------------
# 7. The 3 customers with the fewest orders placed (TOP 3 ASC)
# -------------------------------------------------------------
bottom_3_customers_df = (
    df_fact_sales.join(df_customers, "customer_key", "left")
    .groupBy("customer_key", "first_name", "last_name")
    .agg(countDistinct("order_number").alias("Total_Orders"))
    .orderBy(col("Total_Orders").asc())
    .limit(3)
)

# -------------------------------------------------------------
# 8. USING WINDOW FUNCTION: Bottom 3 customers with tie-breaker
# -------------------------------------------------------------
customer_orders_spec = Window.orderBy(
    col("Total_Orders").asc(), 
    col("customer_key").asc()
)
bottom_3_customers_window_df = (
    df_fact_sales.join(df_customers, "customer_key", "left")
    .groupBy("customer_key", "first_name", "last_name")
    .agg(countDistinct("order_number").alias("Total_Orders"))
    .withColumn("Rank_Customers", row_number().over(customer_orders_spec))
    .filter(col("Rank_Customers") <= 3)
)

# =============================================================
# DISPLAY RESULTS IN SEQUENTIAL SQL ORDER
# =============================================================
print("--- 1. Top 5 Products (LIMIT) ---")
display(top_5_products_df)

print("--- 2. Top 5 Subcategories (LIMIT) ---")
display(top_5_subcategories_df)

print("--- 3. Top 5 Products (WINDOW RANK) ---")
display(top_5_products_window_df)

print("--- 4. 5 Worst-Performing Products ---")
display(worst_5_products_df)

print("--- 5. Top 10 Customers by Revenue (LIMIT) ---")
display(top_10_customers_df)

print("--- 6. Top 10 Customers by Revenue (WINDOW RANK) ---")
display(top_10_customers_window_df)

print("--- 7. Bottom 3 Customers by Orders (LIMIT) ---")
display(bottom_3_customers_df)

print("--- 8. Bottom 3 Customers by Orders (WINDOW ROW_NUMBER) ---")
display(bottom_3_customers_window_df)